In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install japanize-matplotlib lightgbm xgboost optuna catboost six --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 35.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:000:00:010:01m


In [3]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # 日本語表示用
from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [4]:
PATH = '/content/drive/MyDrive/Colab Notebooks/GCI2026_Summer/Competition2/data/'

In [5]:
train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
print('Train:', train.shape, ' Test:', test.shape)

Train: (1568, 22)  Test: (672, 21)


In [6]:
median_income = train['annual_income'].median()
for df in [train, test]:
    df['annual_income'] = df['annual_income'].fillna(median_income)

In [7]:
for col in ['education_level', 'marital_status']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [8]:
for df in [train, test]:
    df.drop(columns=['customer_id', 'registration_date'], inplace=True)
print('Train欠損:', train.isnull().sum().sum(), ' 列:', list(train.columns))

Train欠損: 0  列: ['birth_year', 'education_level', 'marital_status', 'annual_income', 'num_children', 'num_teenagers', 'days_since_last_purchase', 'spend_wines', 'spend_fruits', 'spend_meat', 'spend_fish', 'spend_sweets', 'spend_gold', 'deals_purchases', 'web_purchases', 'catalog_purchases', 'store_purchases', 'monthly_web_visits', 'has_complaint', 'target']


In [9]:
def cv_auc(model, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = m.predict_proba(X.iloc[va_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[va_idx], p))
    return np.mean(scores), scores



In [10]:
spend_cols = ['spend_wines', 'spend_fruits', 'spend_meat', 'spend_fish', 'spend_sweets', 'spend_gold']
for df in [train, test]:
    df['age'] = 2024 - df['birth_year']  # 基準年は便宜上の固定値。大小関係が保てればよい
    df['total_spend'] = df[spend_cols].sum(axis=1)  # spend_* 6列の合計

    df['total_purchases'] = df['web_purchases'] + df['catalog_purchases'] + df['store_purchases']

    # df['web_dependency'] = np.where(df['total_purchases'] == 0, -1, df['web_purchases'] / df['total_purchases'])
    # df['catalog_dependency'] = np.where(df['total_purchases'] == 0, -1, df['catalog_purchases'] / df['total_purchases'])
    df['store_dependency'] = np.where(df['total_purchases'] == 0, -1, df['store_purchases'] / df['total_purchases'])
    
    df['average_order_value'] = np.where(df['total_purchases'] == 0, -1, df['total_spend'] / df['total_purchases'])
    

In [11]:
X = train.drop(columns=['target'])
y = train['target']
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42)
baseline_cv, folds = cv_auc(rf, X, y)

In [12]:
models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42),
    'LogisticRegression': make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1),
    'MLP': make_pipeline(StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
        eval_metric='logloss', random_state=42),
    'CatBoost' : CatBoostClassifier(iterations=300, depth=6, verbose=0, learning_rate=0.1),
}

cv_scores = {}
for name, model in models.items():
    mean_auc, _ = cv_auc(model, X, y)
    cv_scores[name] = mean_auc
    print(f'{name:20s} CV AUC = {mean_auc:.4f}')

RandomForest         CV AUC = 0.8266
LogisticRegression   CV AUC = 0.8108
LightGBM             CV AUC = 0.8263
MLP                  CV AUC = 0.7812
XGBoost              CV AUC = 0.8308
CatBoost             CV AUC = 0.8254


In [13]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_lgb(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 7, 63),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 80),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    model = LGBMClassifier(**params, random_state=42, verbose=-1)
    mean_auc, _ = cv_auc(model, X, y)
    return mean_auc

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)  # 試行回数。増やすほど時間がかかる
print(f'Optuna ベスト CV AUC = {study_lgb.best_value:.4f}  （基準 {baseline_cv:.4f}, 差 {study_lgb.best_value - baseline_cv:+.4f}）')

  0%|          | 0/30 [00:00<?, ?it/s]

Optuna ベスト CV AUC = 0.8449  （基準 0.8266, 差 +0.0184）


In [14]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_cat(trial):
    params = {
        'iterations'          : trial.suggest_int('iterations', 100, 500),
        'depth'               : trial.suggest_int('depth', 4, 8),
        'learning_rate'       : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_strength'     : trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
        'bagging_temperature' : trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'l2_leaf_reg'         : trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'border_count'        : trial.suggest_int('border_count', 32, 255),
        'od_type'             : trial.suggest_categorical('od_type', ['IncToDec', 'Iter']),
        'od_wait'             : trial.suggest_int('od_wait', 10, 50),
    }
    model = CatBoostClassifier(**params, random_state=42, verbose=0)
    mean_auc, _ = cv_auc(model, X, y)
    return mean_auc

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=30, show_progress_bar=True)
print(f'Optuna ベスト CV AUC = {study_cat.best_value:.4f}  （基準 {baseline_cv:.4f}, 差 {study_cat.best_value - baseline_cv:+.4f}）')

  0%|          | 0/30 [00:00<?, ?it/s]

Optuna ベスト CV AUC = 0.8477  （基準 0.8266, 差 +0.0211）


In [15]:
# ベスト設定で 5-fold それぞれ学習し、test 予測を平均する（k-fold アンサンブル）
X_test = test[X.columns]  # train と同じ特徴量・順序に揃える
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_pred = np.zeros(len(test))
for tr_idx, va_idx in skf.split(X, y):
    # m = LGBMClassifier(**study_lgb.best_params, random_state=42, verbose=-1)
    m = CatBoostClassifier(**study_cat.best_params, random_state=42, verbose=0)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    test_pred += m.predict_proba(X_test)[:, 1] / skf.n_splits
print('ベスト設定:', study_cat.best_params)

ベスト設定: {'iterations': 454, 'depth': 5, 'learning_rate': 0.017320136826730753, 'random_strength': 4.077830844495091e-05, 'bagging_temperature': 0.0002503530152348308, 'l2_leaf_reg': 1.8470515432867984, 'border_count': 121, 'od_type': 'Iter', 'od_wait': 44}


In [16]:
# 9章で作った test_pred（5-fold アンサンブルの平均予測）から提出ファイルを作る
submission = pd.read_csv(PATH + 'sample_submission.csv')
submission['target'] = test_pred
submission.to_csv(PATH + 'submission_advanced.csv', index=False)
print('提出行数:', len(submission))
print('target範囲:', round(submission['target'].min(), 3), '〜', round(submission['target'].max(), 3))
submission.head()

提出行数: 672
target範囲: 0.002 〜 0.877


,customer_id,target
0,1568,0.770566
1,1569,0.556954
2,1570,0.173717
3,1571,0.011909
4,1572,0.076285


In [17]:
# Google Colab からダウンロードする場合
from google.colab import files
files.download(PATH + 'submission_advanced.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>